In [1]:
import pandas as pd
import numpy as np
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import shutil

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [2]:
coco_path = '../Data/Raw/training_data/quadrant-enumeration-disease/train_quadrant_enumeration_disease.json'
images_path = '../Data/Raw/training_data/quadrant-enumeration-disease/xrays'

with open(coco_path) as f:
    coco_file = json.load(f)

In [ ]:
s1_train_path = r'..\Data\Processed\Stage 1 (tooth)\train'
s1_val_path = r'..\Data\Processed\Stage 1 (tooth)\val'
s1_test_path = r'..\Data\Processed\Stage 1 (tooth)\test'

In [4]:
diagnosis_map = {
    0: "impacted",
    1: "caries",
    2: "periapical",
    3: "deep_caries"
}

quad_map = {
    0: "Upper Right",
    1: "Upper Left",
    2: "Lower Left",
    3: "Lower Right"
}

def create_df(diseases_dict, quads_dict):

    f_name = []
    bbox = []
    seg = []
    quad = []
    tooth_num = []
    disease = []
    fdi = []

    for item in coco_file['images']:
        for ann in coco_file['annotations']:
            if int(ann['image_id']) == int(item['id']):
                f_name.append(item['file_name'])
                bbox.append(ann['bbox'])
                seg.extend(ann['segmentation'])
                quad.append(quads_dict[ann['category_id_1']])
                tooth_num.append(ann['category_id_2'])
                disease.append(diseases_dict[ann['category_id_3']])
                fdi.append(ann['category_id_1'] * 10 + tooth_num[-1])

    df = pd.DataFrame({'File_Name':f_name,
                    'bbox':bbox,
                    'Seg':seg,
                    'Quad':quad,
                    'Tooth_Num':tooth_num,
                    'FDI': fdi,
                    'Disease_Name':disease})

    return df

df = create_df(diagnosis_map,quad_map)
df

,File_Name,bbox,Seg,Quad,Tooth_Num,FDI,Disease_Name
0,train_673.png,"[542.0, 698.0, 220.0, 271.0]","[621, 703, 573, 744, 542, 885, 580, 945, 650, ...",Lower Right,7,37,impacted
1,train_673.png,"[1952.0, 693.0, 177.0, 270.0]","[2045, 693, 2109, 734, 2129, 915, 2047, 963, 2...",Lower Left,7,27,impacted
2,train_673.png,"[675.0, 708.0, 243.0, 300.0]","[784, 711, 754, 746, 737, 821, 678, 916, 675, ...",Lower Right,6,36,caries
3,train_673.png,"[1463.0, 725.0, 98.0, 425.0]","[1464, 749, 1513, 725, 1550, 760, 1555, 798, 1...",Lower Left,2,22,caries
4,train_673.png,"[1536.0, 753.0, 103.0, 381.0]","[1543, 796, 1590, 753, 1622, 796, 1629, 840, 1...",Lower Left,3,23,caries
...,...,...,...,...,...,...,...
3524,train_370.png,"[1851.2857142857142, 474.2857142857142, 117.14...","[1885, 477, 1868, 597, 1859, 657, 1851, 728, 1...",Upper Left,5,15,caries
3525,train_370.png,"[1959.0, 479.9999999999999, 127.0, 274.2857142...","[2005, 488, 1974, 479, 1965, 522, 1965, 588, 1...",Upper Left,6,16,caries
3526,train_370.png,"[2024.9999999999998, 463.0, 147.00000000000023...","[2064, 463, 2024, 471, 2036, 559, 2056, 628, 2...",Upper Left,7,17,deep_caries
3527,train_370.png,"[1921.7142857142856, 749.0, 221.28571428571445...","[1942, 783, 1924, 806, 1921, 835, 1944, 872, 1...",Lower Left,6,26,caries


In [5]:
df.to_csv(r'..\Data\Processed\data (before spliting).csv')

In [7]:
df['Disease_Name'].value_counts()

Disease_Name
caries         2189
impacted        604
deep_caries     578
periapical      158
Name: count, dtype: int64